模拟三个远程 I/O 请求，耗时分别为 0.2、0.1、0.3 秒；并发执行，结果必须保持输入顺序。

In [1]:
pip install asyncio

Note: you may need to restart the kernel to use updated packages.


In [8]:
import time
import asyncio

async def query(num:int, sleep_time:float):
    await asyncio.sleep(sleep_time)
    return f"Result from query: {num}"

async def main() -> None:
    L = await asyncio.gather(query(1,0.2),query(2,0.1),query(3,0.3))
    print(L)

if __name__ == "__main__":
    await main()

['Result from query: 1', 'Result from query: 2', 'Result from query: 3']


题目 2：并发限流
给定 10 个 ID，同时最多运行 3 个请求。

In [5]:
async def main(ids: list[int]):
    semaphore = asyncio.Semaphore(3)
    async def concurrent_requests(id: int) -> int:
        async with semaphore:
            await asyncio.sleep(1)
            print(f"Request {id} completed")
            return id
    await asyncio.gather(*(concurrent_requests(id) for id in ids))
    
if __name__ == "__main__":
    await main([i for i in range(1, 11)])

Request 1 completed
Request 2 completed
Request 3 completed
Request 4 completed
Request 5 completed
Request 6 completed
Request 7 completed
Request 8 completed
Request 9 completed
Request 10 completed


实现请求模拟函数，超过 0.1 秒返回默认值 None，且不能吞掉与超时无关的异常。

In [7]:

async def query(delay: float, id: int) -> str:
    await asyncio.sleep(delay)
    return f"Query {id} completed after {delay} seconds"

async def main(delay: float, id: int):
    try:
        result = await asyncio.wait_for(query(delay, id), timeout=0.1)
        print(result)
    except TimeoutError:
        print("Query timed out")
        return None

if __name__ == "__main__":
    await main(0.2, 1)
    await main(0.02, 1)

Query timed out
Query 1 completed after 0.02 seconds


用 Python 3.11 的 asyncio.TaskGroup 启动两项工作，收集结果。回答：其中一项异常时另一项会发生什么？